[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/masanorihirano/pams/blob/main/examples/fat_tail.ipynb)

In [ ]:
# Please remove comment-out if necessary
#! pip install pams matplotlib

# Fat tails and volatility clustering

This notebook ports the FatTail sample of plhamJ ([samples/fat_tail](https://github.com/masanorihirano/pams/tree/main/samples/fat_tail) in this repository) and its R scripts. FCN agents without the chartist component trade in one market, and the returns of the market price are checked against two stylized facts of real financial markets:

- **Fat tails**: large returns are much more frequent than for a normal distribution. The kurtosis is higher than that of a normal distribution, and the tail of the distribution decays like a power law.
- **Volatility clustering**: large price changes tend to be followed by large price changes. The absolute returns stay positively autocorrelated for many lags, although the signs of the returns are hardly predictable.

The config is the one of plhamJ with two changes: the second session has 20000 steps instead of 60000 so that the notebook finishes quickly, and the orders use the default fixed margins instead of `"marginType": "normal"` (see the docstring of [samples/fat_tail/main.py](https://github.com/masanorihirano/pams/blob/main/samples/fat_tail/main.py)).

In [ ]:
config = {
	"simulation": {
		"markets": ["Market"],
		"agents": ["FCNAgents"],
		"sessions": [
			{	"sessionName": 0,
				"iterationSteps": 500,
				"withOrderPlacement": True,
				"withOrderExecution": False,
				"withPrint": False
			},
			{	"sessionName": 1,
				"iterationSteps": 20000, "MEMO": "plhamJ runs 60000 steps",
				"withOrderPlacement": True,
				"withOrderExecution": True,
				"withPrint": True
			}
		]
	},

	"Market": {
		"class": "Market",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000
	},

	"FCNAgents": {
		"class": "FCNAgent",
		"numAgents": 1000,

		"MEMO": "Agent class",
		"markets": ["Market"],
		"assetVolume": 50,
		"cashAmount": 10000,

		"MEMO": "FCNAgent class",
		"fundamentalWeight": {"expon": [1.0]},
		"chartWeight": {"expon": [0.0]},
		"noiseWeight": {"expon": [1.0]},
		"noiseScale": 0.001,
		"timeWindowSize": [100, 500],
		"orderMargin": [0.0, 0.1],
		"marginType": "fixed", "MEMO": "plhamJ uses normal (see samples/fat_tail/main.py)"
	}
}

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pams.runners import SequentialRunner
from pams.logs.market_step_loggers import MarketStepSaver

In [ ]:
saver = MarketStepSaver()

runner = SequentialRunner(
    settings=config,
    prng=random.Random(42),
    logger=saver,
)
runner.main()

## Market price

In [ ]:
market_prices = dict(sorted(map(lambda x: (x["market_time"], x["market_price"]), saver.market_step_logs)))
fundamental_prices = dict(sorted(map(lambda x: (x["market_time"], x["fundamental_price"]), saver.market_step_logs)))

In [ ]:
plt.figure()
plt.plot(list(market_prices.keys()), list(market_prices.values()), linewidth=0.5, color="#2a78d6", label="market price")
plt.plot(list(fundamental_prices.keys()), list(fundamental_prices.values()), color="black", label="fundamental price")
plt.xlabel("ticks")
plt.ylabel("price")
plt.legend()
plt.show()

## Returns

As in the R scripts of plhamJ, the returns are the log returns of the market price in the second session (the one with order execution). The first steps of the session are dropped as a transient, and so are the steps where the price did not change. The normalized returns are divided by their standard deviation.

In [ ]:
burn_in = 2000  # plhamJ drops the first 10000 of its 60000 steps

session_prices = dict(sorted(map(lambda x: (x["market_time"], x["market_price"]), filter(lambda x: x["session_id"] == 1, saver.market_step_logs))))
prices = np.array(list(session_prices.values()))[burn_in:]
prices = prices[np.insert(prices[1:] != prices[:-1], 0, True)]  # drop unchanged prices
returns = np.diff(np.log(prices))
normalized_returns = returns / np.std(returns, ddof=1)
print("number of returns:", len(returns))

## Summary statistics

The statistics of `statistics.R` of plhamJ. The kurtosis is the excess kurtosis, which is 0 for a normal distribution; a large positive value means fat tails.

In [ ]:
print("mean:", np.mean(returns))
print("var:", np.var(returns, ddof=1))
print("sd:", np.std(returns, ddof=1))
print("skewness:", stats.skew(returns))
print("excess kurtosis:", stats.kurtosis(returns))

## Return distribution

The histogram of the normalized returns on a log scale, with the density of the standard normal distribution (`fattail.R` of plhamJ), and a QQ plot against the normal distribution. With fat tails, the histogram stays far above the normal density away from 0, and the QQ plot bends away from the diagonal at both ends.

In [ ]:
r_max = np.max(np.abs(normalized_returns))
density, edges = np.histogram(normalized_returns, bins=np.append(np.arange(-r_max, r_max, 0.5), r_max), density=True)
mids = (edges[:-1] + edges[1:]) / 2
x = np.linspace(-r_max, r_max, 1000)
plt.figure()
plt.plot(mids[density > 0], density[density > 0], "o", color="#2a78d6", label="simulation")
plt.plot(x, stats.norm.pdf(x), color="black", label="normal distribution")
plt.yscale("log")
plt.ylim([np.min(density[density > 0]) / 10, np.max(density) * 10])
plt.xlabel("return / sd(return)")
plt.ylabel("density")
plt.legend()
plt.show()

In [ ]:
(theoretical_quantiles, ordered_returns), _ = stats.probplot(normalized_returns, dist="norm")
plt.figure()
plt.plot(theoretical_quantiles, ordered_returns, "o", markersize=3, color="#2a78d6", label="simulation")
plt.plot(theoretical_quantiles, theoretical_quantiles, color="black", label="normal distribution")
plt.xlabel("quantile of the standard normal distribution")
plt.ylabel("quantile of return / sd(return)")
plt.legend()
plt.show()

## Tail index

If the tail follows a power law $P(|r| > x) \propto x^{-\alpha}$, the complementary cumulative distribution (CCDF) of the absolute returns is a straight line with slope $-\alpha$ on log-log axes, whereas the tail of the normal distribution falls off much faster. The Hill estimator estimates the tail index $\alpha$ from the $k$ largest absolute returns $x_{(1)} \geq x_{(2)} \geq \cdots$:

$$\hat{\alpha}_k = \left( \frac{1}{k} \sum_{i=1}^{k} \log x_{(i)} - \log x_{(k+1)} \right)^{-1}.$$

For real stock returns, $\alpha$ is typically around 3 (the inverse cubic law). As a reference, the Hill estimates of a sample from the normal distribution of the same size are also plotted; they keep growing as $k$ decreases because the normal distribution has no power-law tail, whereas those of the simulation stay much lower.

In [ ]:
abs_returns = np.sort(np.abs(normalized_returns))[::-1]
ccdf = np.arange(1, len(abs_returns) + 1) / len(abs_returns)
x = np.logspace(-1, np.log10(abs_returns[0]), 200)
plt.figure()
plt.loglog(abs_returns, ccdf, color="#2a78d6", label="simulation")
plt.loglog(x, 2 * stats.norm.sf(x), color="black", label="normal distribution")
plt.xlim(left=0.1)
plt.ylim([0.5 / len(abs_returns), 1.5])
plt.xlabel("|return| / sd(return)")
plt.ylabel("P(|return| / sd(return) > x)")
plt.legend()
plt.show()

In [ ]:
def hill_estimates(values, max_k):
    log_values = np.log(np.sort(np.abs(values))[::-1])
    ks = np.arange(1, max_k + 1)
    return ks, 1.0 / (np.cumsum(log_values)[:max_k] / ks - log_values[1 : max_k + 1])

max_k = len(returns) // 10
ks, hill = hill_estimates(returns, max_k)
_, hill_normal = hill_estimates(np.random.default_rng(42).standard_normal(len(returns)), max_k)
plt.figure()
plt.plot(ks, hill, color="#2a78d6", label="simulation")
plt.plot(ks, hill_normal, color="black", label="normal distribution")
plt.axhline(3, color="gray", linestyle="--", label="inverse cubic law")
plt.ylim([0, 10])
plt.xlabel("k (number of the largest absolute returns)")
plt.ylabel("Hill estimate of the tail index")
plt.legend()
plt.show()
print("Hill estimate with the largest 1% of the absolute returns:", hill[len(returns) // 100 - 1])
print("... of the normal sample:", hill_normal[len(returns) // 100 - 1])

## Volatility clustering

The autocorrelation of the returns and of the absolute returns (`volcluster.R` of plhamJ). The dashed lines are the approximate 95% bounds $\pm 1.96 / \sqrt{n}$ for uncorrelated values. The strongly negative autocorrelation of the returns at lag 1 is the bid-ask bounce: the execution price jumps between the best bid and the best ask. Beyond the first few lags the returns are almost uncorrelated, whereas the absolute returns stay positively correlated for tens of lags.

In [ ]:
def autocorrelation(values, max_lag):
    deviations = values - np.mean(values)
    return np.array([np.sum(deviations[:-lag] * deviations[lag:]) for lag in range(1, max_lag + 1)]) / np.sum(deviations ** 2)

max_lag = 50
lags = np.arange(1, max_lag + 1)
bound = 1.96 / np.sqrt(len(returns))
plt.figure()
plt.plot(lags, autocorrelation(returns, max_lag), "o-", color="#2a78d6", label="return")
plt.plot(lags, autocorrelation(np.abs(returns), max_lag), "o-", color="#eb6834", label="absolute return")
plt.axhline(0, color="black", linewidth=0.5)
plt.axhline(bound, color="gray", linestyle="--")
plt.axhline(-bound, color="gray", linestyle="--")
plt.xlabel("lag")
plt.ylabel("autocorrelation")
plt.legend()
plt.show()

## Scaling up

For the scale of plhamJ, set `iterationSteps` of the second session to 60000 and `burn_in` to 10000. The run then takes about three times longer, and the estimates, in particular the Hill estimates for small $k$, become more stable.